# Notebook 2: Time Series Analysis

## Overview

This notebook conducts time series diagnostics on the MENA GDP growth panel prior to model estimation. The analysis covers:

1. **Stationarity testing**: ADF and KPSS tests for each country series
2. **Seasonal decomposition**: trend, seasonal, and residual components
3. **Autocorrelation analysis**: ACF and PACF at 20 lags

Results inform ARIMA order selection and confirm whether differencing is required.

---

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from statsmodels.tsa.stattools import adfuller, kpss

from src.data_processing import (
    MENA_COUNTRIES, get_country_series, train_test_split_series
)
from src.visualization import (
    plot_decomposition, plot_acf_pacf, plot_stationarity_summary
)

panel = pd.read_csv('../data/processed/mena_gdp_panel.csv')

## Stationarity Tests

Two complementary tests are applied to each country series:

- **ADF (Augmented Dickey-Fuller)**: H₀ = unit root (non-stationary). Rejection at 5% indicates stationarity.
- **KPSS (Kwiatkowski-Phillips-Schmidt-Shin)**: H₀ = stationary. Rejection at 5% indicates non-stationarity.

A series is classified as stationary when ADF rejects the null and KPSS does not, and as non-stationary when the reverse holds. Conflicting results are treated conservatively (non-stationary).


In [ ]:
stationarity_results = []

for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()

    adf_stat, adf_p, *_ = adfuller(series, autolag='AIC')
    adf_stationary = adf_p < 0.05

    try:
        kpss_stat, kpss_p, *_ = kpss(series, regression='c', nlags='auto')
        kpss_stationary = kpss_p >= 0.05
    except Exception:
        kpss_stat, kpss_p, kpss_stationary = np.nan, np.nan, None

    stationarity_results.append({
        'Country': name,
        'ADF Stat': round(adf_stat, 3),
        'ADF p-value': round(adf_p, 4),
        'ADF Stationary': adf_stationary,
        'KPSS Stat': round(kpss_stat, 3) if not np.isnan(kpss_stat) else np.nan,
        'KPSS p-value': round(kpss_p, 4) if not np.isnan(kpss_p) else np.nan,
        'KPSS Stationary': kpss_stationary,
    })

stat_df = pd.DataFrame(stationarity_results)
stat_df.to_csv('../results/tables/stationarity_tests.csv', index=False)
print(stat_df[['Country', 'ADF p-value', 'ADF Stationary', 'KPSS p-value', 'KPSS Stationary']].to_string(index=False))

In [ ]:
plot_stationarity_summary(
    stat_df,
    save_path='../results/figures/02_stationarity_heatmap.png',
)

## Seasonal Decomposition

Additive decomposition is applied to each series to separate the long-run trend from cyclical and residual variation. For annual data, the period is set to 1 and decomposition extracts a smoothed trend component.


In [ ]:
for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    plot_decomposition(
        series, name,
        save_path=f'../results/figures/02_decomposition_{code}.png',
    )

## Autocorrelation Analysis

ACF and PACF plots at 20 lags are inspected for each country to identify significant autocorrelation structures. Significant spikes in the PACF at lag p and cut-off in the ACF suggest an AR(p) process; the reverse pattern suggests an MA(q) process.


In [ ]:
for code, name in MENA_COUNTRIES.items():
    series = get_country_series(panel, code).dropna()
    plot_acf_pacf(
        series, name, lags=20,
        save_path=f'../results/figures/02_acf_pacf_{code}.png',
    )

## Summary

Most MENA GDP growth series exhibit stationarity (or near-stationarity) at levels, consistent with mean-reverting growth dynamics around a slowly shifting trend. Several series show significant autocorrelation at lag 1, motivating low-order ARIMA and ETS models. Notebook 3 proceeds with estimation of the classical forecasting models.
